#### **1. INITIALIZATION**

In [1]:
import os
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#### **2. SCAN THROUGH ALL FOLDERS, GET IMAGE FILE INFOS**

In [3]:
# 1. Base directory containing all your data
base_path = "/content/drive/MyDrive/TrainingData"
data_records = []
real_image_dataset = ["vision","nature"]

In [ ]:
# 2. Traverse the directory tree using os.walk
for root, dirs, files in os.walk(base_path):
    for file in files:
        # Filter for image files only
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):

            # Create absolute file path
            full_path = os.path.join(root, file)

            # Label is the direct parent folder of the image (Sub-Folder)
            label_name = os.path.basename(root)

            # Extract relative path to get the first-level folder under TrainingData
            rel_path = os.path.relpath(root, base_path)
            dataset_name = rel_path.split(os.sep)[0]

            # Check if the extracted dataset_name exists in your predefined list
            is_real = 1 if (dataset_name in real_image_dataset) or (label_name in real_image_dataset) else 0

            # Append extracted information to the list
            data_records.append({
                "file_name": file,
                "dataset": dataset_name,
                "label": label_name,
                "real_image": is_real,
                "file_path": full_path
            })

In [ ]:
# 3. Convert the list of records into a Pandas DataFrame
df = pd.DataFrame(data_records)

# 4. Print summary report
print(f"Scan complete! Total images found: {len(df)}")

# Display the first 5 rows to verify the columns
display(df.head())

# 5. Save the DataFrame to a CSV file for future use
csv_path = "/content/drive/MyDrive/TrainingData/dataset_inventory.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

df.to_csv(csv_path, index=False)
print(f"\nData inventory successfully saved to: {csv_path}")

Scan complete! Total images found: 28377


,file_name,dataset,label,real_image,file_path
0,D34_I_flat_0001.jpg,vision,D34_Apple_iPhone5,1,/content/drive/MyDrive/TrainingData/vision/D34...
1,D34_I_flat_0002.jpg,vision,D34_Apple_iPhone5,1,/content/drive/MyDrive/TrainingData/vision/D34...
2,D34_I_flat_0003.jpg,vision,D34_Apple_iPhone5,1,/content/drive/MyDrive/TrainingData/vision/D34...
3,D34_I_flat_0004.jpg,vision,D34_Apple_iPhone5,1,/content/drive/MyDrive/TrainingData/vision/D34...
4,D34_I_flat_0005.jpg,vision,D34_Apple_iPhone5,1,/content/drive/MyDrive/TrainingData/vision/D34...



Data inventory successfully saved to: /content/drive/MyDrive/TrainingData/dataset_inventory.csv


#### **3. SPLIT DATA INTO TRAIN/VAL/TEST**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the existing dataset inventory
csv_path = "/content/drive/MyDrive/TrainingData/dataset_inventory.csv"
df = pd.read_csv(csv_path)

print(f"Total images before splitting: {len(df)}")

# 2. First Split: 80% Train, 20% Temporary (Val + Test)
# The 'stratify' parameter ensures the Real/Fake ratio remains balanced across splits
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df['real_image'],
    random_state=42 # Fixed seed for reproducibility
)

# 3. Second Split: Divide the Temporary 20% into 10% Val and 10% Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['real_image'],
    random_state=42
)

# 4. Create a new column 'split' and assign values based on the indices
df.loc[train_df.index, 'split'] = 'train'
df.loc[val_df.index, 'split'] = 'val'
df.loc[test_df.index, 'split'] = 'test'

# 5. Print a verification report
print("\nData Split Distribution:")
print(df.groupby(['split', 'real_image']).size().unstack(fill_value=0))

# 6. Save the updated DataFrame back to the CSV
df.to_csv(csv_path, index=False)
print(f"\nSuccessfully added 'split' column and saved to {csv_path}")

Total images before splitting: 28377

Data Split Distribution:
real_image      0      1
split                   
test         1400   1438
train       11201  11500
val          1400   1438

Successfully added 'split' column and saved to /content/drive/MyDrive/TrainingData/dataset_inventory.csv


#### **4. SAMPLE FROM DATASET**

In [ ]:
import os
import shutil
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

# ========================================================
# GIAI ĐOẠN 1: LẤY MẪU VÀ ĐÁNH DẤU VÀO FILE GỐC
# ========================================================

# 1. Đọc file CSV chứa toàn bộ 28k ảnh
csv_path = "/content/drive/MyDrive/TrainingData/dataset_inventory.csv"
df = pd.read_csv(csv_path)

# 2. Hàm lấy mẫu 5000 ảnh (4000 Train, 500 Val, 500 Test)
# Việc bốc ngẫu nhiên (sample) từ các tập đã chia sẽ tự động giữ được sự cân bằng giữa ảnh Real/Fake
def get_sample(group):
    if group.name == 'train':
        return group.sample(n=4000, random_state=42)
    elif group.name == 'val':
        return group.sample(n=500, random_state=42)
    elif group.name == 'test':
        return group.sample(n=500, random_state=42)
    return group

# Lấy ra các index của 5000 ảnh được chọn
sampled_df = df.groupby('split', group_keys=False).apply(get_sample)
sampled_indices = sampled_df.index

# 3. Tạo cột 'sample_data': Đánh dấu 1 cho 5000 ảnh được chọn, 0 cho các ảnh còn lại
df['sample_data'] = 0
df.loc[sampled_indices, 'sample_data'] = 1

# Lưu lại file CSV gốc để ghi nhận cột mới
df.to_csv(csv_path, index=False)
print(f"✅ Đã đánh dấu {len(sampled_indices)} ảnh vào cột 'sample_data' và lưu đè lên file: {csv_path}")


# ========================================================
# GIAI ĐOẠN 2: COPY DATA ĐƯỢC ĐÁNH DẤU XUỐNG LOCAL DISK
# ========================================================

local_sample_dir = "/content/Local_5K_Sample"
local_sample_csv = "/content/local_5k_inventory.csv"
os.makedirs(local_sample_dir, exist_ok=True)

# 4. Lọc ra đúng 5000 ảnh đã được đánh dấu
df_sample_only = df[df['sample_data'] == 1].copy()

# Hàm copy từng file
def copy_single_image(row_data):
    index, row = row_data
    src_path = row['file_path']

    # Tạo tên file an toàn (thêm index vào trước để tránh trùng tên file từ các thư mục khác nhau)
    safe_filename = f"{index}_{row['file_name']}"
    dest_path = os.path.join(local_sample_dir, safe_filename)

    # Copy nếu chưa tồn tại
    if not os.path.exists(dest_path):
        try:
            shutil.copy2(src_path, dest_path)
        except Exception as e:
            return None # Bỏ qua nếu file gốc bị lỗi hoặc không tồn tại

    return dest_path

print(f"🚀 Đang copy {len(df_sample_only)} ảnh sang Local Disk ({local_sample_dir})...")
rows = list(df_sample_only.iterrows())

# 5. Dùng đa luồng (Multi-threading) để copy siêu tốc (thay vì copy từng file một)
with ThreadPoolExecutor(max_workers=16) as executor:
    new_paths = list(tqdm(executor.map(copy_single_image, rows), total=len(df_sample_only)))

# 6. Cập nhật cột file_path trỏ về ổ Local
df_sample_only['file_path'] = new_paths
df_sample_only = df_sample_only.dropna(subset=['file_path']) # Xóa các dòng bị lỗi không copy được

# Lưu thành 1 file CSV mới (Local CSV) chuyên dùng để Train siêu tốc
df_sample_only.to_csv(local_sample_csv, index=False)
print(f"🎉 Hoàn tất! File CSV local chuẩn bị cho DataLoader đã lưu tại: {local_sample_csv}")

/tmp/ipython-input-214/745399371.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df.groupby('split', group_keys=False).apply(get_sample)


✅ Đã đánh dấu 5000 ảnh vào cột 'sample_data' và lưu đè lên file: /content/drive/MyDrive/TrainingData/dataset_inventory.csv
🚀 Đang copy 5000 ảnh sang Local Disk (/content/Local_5K_Sample)...


  0%|          | 0/5000 [00:00<?, ?it/s]